# Message Queues

Message queues offer a solution for scenarios where an application server faces a high volume of request that cannot be processed immediately. Instead of scaling horizontally or vertically, it is not always cost-efficient or practical. And in some intances where immediate processing of these requests isnt neccessary, allowing them to be queued for later handling. Message queues serve to decouple producer (app events) and consumers (application servers), functioning as a buffer for manaing surges in data.

### Example

Handling payment process is a good illustration of a system where message queues can provide benefits:
- 1. Handling peaks in load: during high usage periods, such as major sale, the number of payment requests typically increases significantly. If these requests were processed sync, it could result in a poor user experience due to prolonged wait times and potential timeouts. However, with message queues, payments can be stored and processed asynchronously.

- 2. Decoupling Service: When a new order is placed, a message can be published to the queue. The payment service, acting as a subsccirber, can then proceess the payment and update the order status.


### Synchronous vs Asynchronous (Queue)

In a synchronous system, the user sends a request and must wait for the entire operation (e.g. payment processing) to complete before receiving a response. This means the user is blocked during processing, which can lead to longer wait times and potential timeouts under high load.

In a message queue-based (asynchronous) system, the user’s request is acknowledged immediately after being placed into the queue, while the actual processing happens later. Although the request is still waiting to be processed in the queue, this waiting occurs within the system rather than blocking the user. This improves user experience and system stability, as requests can be buffered and processed gradually without overwhelming the application servers.

There are different methods which a message queue can interact with the application server.

### Push/Pull Model (How messages are delivered)

- 1. Pull-Based Model: The application is responsible for monitoring the message queue for any new messages. If new messages are present and the app has capacity, it "pulls" from the queue. This approach can be more efficient in terms of managing the server-side load. However, if the queue is empty, it may introduce latency.

<img src="image/pull.png" width="600">


- 2. Push-Based Model: In this case, the queue takes on responsibility of pushing messages to the server. But, this strategy might overload the server if the rate of incoming messages is very high. When the message queue dispatches the message to the application server, the server sends an acknowledgement after successfully processing a message. If it does not, it can infer the message was not processed, prompting the queue to resend it.

<img src="image/push.png" width="600">


### Pub/Sub Model (What alternative system to Queues)

This model allows decoupling of the publisher and subscribers, eliminating the need for either to be aware of each other's existence. This allows for easy system scalaibiliy and ensures messages are not lost if a subscriber is temporarily unable to process them.

The steps are:
- 1. The publisher dispatches messages to a speicifc queue or topic
- 2. One or more subscriber listens to the specific queue or topic
- 3. The message broker ensures all messages published to a topic are successfully delivered to all subscribers of that topic. Subscribers process messages independelty and at their own pace.

Publishers dispatch their message to specific topics, and subscribers indicate their interest by subscribing to these topics. A "topic" is a category or label that servers as a group for similar messages, it helps categorize the majority of the messages that can be recieved. Another beneift of the pub/sub model is that we can introudce a completely different API as a subscriber without changing the architecture. Therefore, new subscribers can be added to a topic without modifying the publishers. This makes the system more flexible and adaptable to chanign requirements.

<img src="image/pub_sub.png" width="700">

In the Pub/Sub model, the publisher does not directly know whether a message has been successfully processed by subscribers. The publisher only receives acknowledgement from the message broker that the message has been accepted. The responsibility of ensuring delivery and handling retries lies with the message broker, while subscribers independently acknowledge and process messages.

Some popular open source are RabbitMQ and Kafka. Cloud based ones by GCP is pub/sub

## MapReduce

It relates to big data processing, enabiling the handling of extensive amount of data, performing computations on it and genreating results. Imagine having billion rows of data containing people's names and theri NRIC Number, and our goal is to process this information such that we only retain the names while removing the SSN. Accomplishing this efficiently would be a task for Map-Reduce.

MapReduce is a programming model coupled with a specific implmentation designed specifically for processing and genreating large datasets. This model is particularly beneficial for distributed computing on vast sets of data spanning terabytes or even petabytes.

Typical Methods that data is being processed

### Batch Processing

In this model, data is processed in substantial groups, or batches. Here, all data is accumulated over a specified period and subsequently processed as a unit. A practical example would be counting the frequency of each word occuring in a book or a series of books. Batch processing doesn't occur in real-time: instead, it takes place when batch jobs are executed. The frequency of these jobs might range from weekly, like generating weekly reports, to daily for producing daily reports.

### Stream Processing

This involves processing data in real-time as it is recieved. Instead of being stored, data is procesed individually in its raw, unbatched form. An example of this could be redacting a customer's credit card expiration date or last name upon payment. This task cannot be performed in a batch and msut be done in real-time, since this information needs to be immediately updated for subsequent operations.

### MapReduce Framework

In this MapReduce framework such as **Apache Hadoop**, the system typically consists of one `master` node and multiple `worker/slave` nodes.

- 1. Master Node: This node is tasked with managing the distribution of the MapReduce job across the worker nodes. It keeps an eye on the status of each task and re-assigns tasks if any failures occur.
- 2. Worker Nodes: THese nodes are whwere the data processing takes place. The master node assigns each worker node a portion of the data and a copy of the MapReduce program.
- 3. Map Phase: Each worker node excutes the Map operation on its own assigned data portion. In our example, this would be mapping each word to a key-value pair where the key is the word and the value is the frequency of the word.
- 4. Shuffle and Sort Phase: After the Map phase, the worker nodes re-organizes the key-value pairs so that all values with the same keys are grouped together. This is known as the shuffle and sort phase. SO for example the given work "The" if workere one processed 3, worker two processed 7 and worker three processed 100, these would be grouped together during this phase.
-5. Reduce Phase: This operation is performed on each group of values, producing a final count for each word. This result is then written to some form of storage or database.

<img src="image/map_reduce.png" width="1000">
